In [2]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv('../data/raw/santander-customer-transaction-prediction/train.csv')
X = df.drop(["target", "ID_code"], axis=1)
y = df["target"]
df.shape
df["target"].value_counts()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [3]:
print(df.shape)
print(df["target"].value_counts())

(200000, 202)
target
0    179902
1     20098
Name: count, dtype: int64


In [6]:
print(y_train.mean(), y_val.mean())


0.1004875 0.1005


In [4]:
import lightgbm as lgb

print(lgb.__version__)
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=100)
    ]
)

4.6.0
[LightGBM] [Info] Number of positive: 16078, number of negative: 143922
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.103883 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51000
[LightGBM] [Info] Number of data points in the train set: 160000, number of used features: 200
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.100487 -> initscore=-2.191820
[LightGBM] [Info] Start training from score -2.191820
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.844877	valid_0's binary_logloss: 0.26224
[200]	valid_0's auc: 0.868673	valid_0's binary_logloss: 0.240771
[300]	valid_0's auc: 0.878555	valid_0's binary_logloss: 0.229269
[400]	valid_0's auc: 0.883851	valid_0's binary_logloss: 0.222258
[500]	valid_0's auc: 0.886695	valid_0's binary_logloss: 0.217955
[600]	valid_0's auc: 0.888077	valid_0's binary_logloss: 0.215295
[700]	valid_0's auc: 0.889455	valid_0's binar

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [7]:
from sklearn.metrics import roc_auc_score

preds = model.predict_proba(X_val)[:,1]
roc_auc_score(y_val, preds)

0.8905518958846678

In [ ]:
test_df = pd.read_csv('../data/raw/santander-customer-transaction-prediction/test.csv')
print(test_df.shape)
test_df = test_df.drop("ID_code", axis=1)

(200000, 201)


In [ ]:
test_preds = model.predict_proba(test_df)[:,1]

In [14]:
submission = pd.read_csv("../data/raw/santander-customer-transaction-prediction/sample_submission.csv")
submission["target"] = test_preds
submission.to_csv("../submissions/baseline.csv", index=False)

In [15]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

In [17]:
importance.head(20)

,feature,importance
174,var_174,247
166,var_166,245
78,var_78,242
190,var_190,237
146,var_146,236
53,var_53,235
133,var_133,235
34,var_34,235
76,var_76,234
22,var_22,233


In [18]:
importance.describe()

,importance
count,200.000000
mean,143.400000
std,56.008345
min,41.000000
25%,100.750000
50%,144.000000
75%,192.250000
max,247.000000


In [19]:
(importance["importance"] == 0).sum()

np.int64(0)